# Smart MCQ Solver — TF-IDF Per-Option Ranker (dual-branch, from scratch, PyTorch)

**Different formulation from the multiclass notebook.** Instead of one 5-way softmax per question, this
treats every `(question, option)` pair as its own row and trains a **binary classifier**: *is this specific
option correct?* At inference, score all 5 options for a question and rank by score for MAP@3.

**Why this can beat multiclass classification on this exact metric:** multiclass cross-entropy only cares
that the *correct* option gets the highest softmax mass -- it has no term that directly rewards getting the
*2nd and 3rd* best guesses right, even though MAP@3 pays for exactly that. A per-option ranker learns
"how correct does this option look on its own" for every option independently, which naturally produces a
usable ranking, not just a single argmax.

**Trade-off to know for your viva:** 5x more rows (more training signal, good for small datasets) but each
row only sees one option's text, so the model has to learn what "looks correct" without directly comparing
options side by side the way the multiclass model's shared input does. Neither formulation strictly dominates
-- that's exactly why we run both and let val MAP@3 decide.


### Fix: `AcceleratorError: no kernel image is available for execution on the device`
On Kaggle this is usually `pip install <package>` silently swapping in a torch build that doesn't match the
driver. Fix: install with `--no-deps`, and verify the GPU with a real op (not just `is_available()`, which
returns `True` even when the installed kernels can't run on the card).


In [2]:
!pip install -q --no-deps wandb

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("torch:", torch.__version__, "| built for CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def get_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        (torch.zeros(1, device="cuda") + 1).cpu()
        return torch.device("cuda")
    except Exception as e:
        print("CUDA reports available but a test op failed, falling back to CPU:", e)
        return torch.device("cpu")

device = get_device()
print("Using device:", device)


torch: 2.10.0+cu128 | built for CUDA: 12.8
GPU: Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


CUDA reports available but a test op failed, falling back to CPU: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Using device: cpu


In [3]:
CONFIG = {
    "word_max_features": 40000,
    "word_ngram_range": (1, 3),
    "word_min_df": 2,
    "word_max_df": 0.9,
    "char_max_features": 20000,
    "char_ngram_range": (3, 5),
    "char_min_df": 2,
    "char_max_df": 0.95,
    "word_hidden": 512,
    "char_hidden": 256,
    "merged_hidden": 256,
    "dropout": 0.3,
    "label_smoothing": 0.05,      # manual smoothing: 1 -> 0.975, 0 -> 0.025
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 256,             # bigger than the multiclass run -- 5x more rows, same question count
    "epochs": 40,
    "early_stop_min_epochs": 12,
    "early_stop_patience": 8,
    "early_stop_min_delta": 0.001,
    "val_split": 0.15,
}

wandb.login()
wandb.init(project="smart-mcq-solver", name="tfidf-pairwise-ranker", config=CONFIG)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


### Melt each question into 5 rows: one per option
`melt_options` is written as 5 vectorized column ops (one per letter), not a Python loop over rows -- keeps
it fast and pandas-idiomatic. Each output row is one `(question, option)` pair with its own binary label.


In [4]:
train_df = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv")

LABELS = ["A", "B", "C", "D", "E"]

def melt_options(df):
    frames = []
    for letter in LABELS:
        sub = pd.DataFrame({
            "qid": df["id"],
            "option": letter,
            "text": "Question: " + df["prompt"].fillna("") + " Answer: " + df[letter].fillna(""),
        })
        if "answer" in df.columns:
            sub["label"] = (df["answer"] == letter).astype(int)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

train_long = melt_options(train_df)   # has 'label'
test_long  = melt_options(test_df)    # no 'label' -- test.csv has no answer column

print("train_long:", train_long.shape, "(", train_df.shape[0], "questions x 5 )")
train_long.head(6)


train_long: (10000, 4) ( 2000 questions x 5 )


,qid,option,text,label
0,1,A,Question: Pick the best possible answer: What ...,0
1,2,A,Question: What is accelerator-based light-ion ...,1
2,3,A,Question: Determine the correct option: What i...,0
3,4,A,Question: Select the most accurate option: Wha...,0
4,5,A,Question: Identify the correct statement: What...,1
5,6,A,Question: Identify the correct statement: What...,0


### TF-IDF (word + char), and a **group-aware** split
Critical detail: we split by `qid`, not by row. If we split rows randomly, 4 of a question's 5 options could
land in train and 1 in val -- the model would implicitly see that question during training and val MAP@3
would be optimistic. `GroupShuffleSplit` guarantees all 5 rows of a question go to the same side.


In [5]:
word_tfidf = TfidfVectorizer(
    analyzer="word", max_features=CONFIG["word_max_features"], ngram_range=CONFIG["word_ngram_range"],
    min_df=CONFIG["word_min_df"], max_df=CONFIG["word_max_df"], sublinear_tf=True, stop_words="english",
)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb", max_features=CONFIG["char_max_features"], ngram_range=CONFIG["char_ngram_range"],
    min_df=CONFIG["char_min_df"], max_df=CONFIG["char_max_df"], sublinear_tf=True,
)

all_text = pd.concat([train_long["text"], test_long["text"]])
word_tfidf.fit(all_text)
char_tfidf.fit(all_text)

Xw_all,  Xc_all  = word_tfidf.transform(train_long["text"]), char_tfidf.transform(train_long["text"])
Xw_test, Xc_test = word_tfidf.transform(test_long["text"]),  char_tfidf.transform(test_long["text"])
y_all = train_long["label"].values

gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG["val_split"], random_state=SEED)
idx_train, idx_val = next(gss.split(train_long, groups=train_long["qid"]))

Xw_train, Xw_val = Xw_all[idx_train], Xw_all[idx_val]
Xc_train, Xc_val = Xc_all[idx_train], Xc_all[idx_val]
y_train,  y_val  = y_all[idx_train],  y_all[idx_val]
meta_val = train_long.iloc[idx_val][["qid", "option", "label"]].reset_index(drop=True)

assert set(train_long.iloc[idx_train]["qid"]).isdisjoint(set(train_long.iloc[idx_val]["qid"])), \
    "leakage: a qid appears on both sides of the split"

print("Word vocab:", len(word_tfidf.vocabulary_), "| Char vocab:", len(char_tfidf.vocabulary_))
print("Train rows:", Xw_train.shape[0], "| Val rows:", Xw_val.shape[0], "(", meta_val['qid'].nunique(), "val questions )")


Word vocab: 24198 | Char vocab: 20000
Train rows: 8500 | Val rows: 1500 ( 300 val questions )


In [6]:
class DualTfidfDataset(Dataset):
    def __init__(self, X_word, X_char, y=None):
        self.X_word, self.X_char, self.y = X_word, X_char, y

    def __len__(self):
        return self.X_word.shape[0]

    def __getitem__(self, idx):
        w = torch.from_numpy(self.X_word[idx].toarray().squeeze(0)).float()
        c = torch.from_numpy(self.X_char[idx].toarray().squeeze(0)).float()
        if self.y is not None:
            return w, c, torch.tensor(self.y[idx], dtype=torch.float32)
        return w, c

train_loader = DataLoader(DualTfidfDataset(Xw_train, Xc_train, y_train), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(DualTfidfDataset(Xw_val, Xc_val, y_val),       batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(DualTfidfDataset(Xw_test, Xc_test),            batch_size=CONFIG["batch_size"], shuffle=False)


In [7]:
class DualBranchScorer(nn.Module):
    """Same dual-branch idea as the multiclass model, but ends in ONE logit: how correct does this
    (question, option) pair look?"""
    def __init__(self, word_vocab, char_vocab, word_hidden, char_hidden, merged_hidden, dropout=0.3):
        super().__init__()
        self.word_branch = nn.Sequential(
            nn.Linear(word_vocab, word_hidden), nn.BatchNorm1d(word_hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.char_branch = nn.Sequential(
            nn.Linear(char_vocab, char_hidden), nn.BatchNorm1d(char_hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(word_hidden + char_hidden, merged_hidden),
            nn.BatchNorm1d(merged_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(merged_hidden, 1),   # single logit: correctness score, not a 5-way distribution
        )

    def forward(self, x_word, x_char):
        w = self.word_branch(x_word)
        c = self.char_branch(x_char)
        return self.head(torch.cat([w, c], dim=1)).squeeze(1)   # (batch,)

model = DualBranchScorer(
    word_vocab=Xw_train.shape[1], char_vocab=Xc_train.shape[1],
    word_hidden=CONFIG["word_hidden"], char_hidden=CONFIG["char_hidden"], merged_hidden=CONFIG["merged_hidden"],
    dropout=CONFIG["dropout"],
).to(device)

wandb.watch(model, log="all", log_freq=50)
print(model)


DualBranchScorer(
  (word_branch): Sequential(
    (0): Linear(in_features=24198, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (char_branch): Sequential(
    (0): Linear(in_features=20000, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=1, bias=True)
  )
)


### MAP@3 has to be computed per-question, not per-row
A batch of rows mixes options from many different questions, so "top-3 of this batch" is meaningless.
We score every validation row once per epoch, attach the scores back to their `qid`/`option`, then rank
within each question group -- this is exactly what happens at real inference time too.


In [8]:
def compute_map3(df):
    """df needs columns: qid, option, score, label (1 row per option, 5 rows per qid)."""
    scores = []
    for _, g in df.groupby("qid"):
        ranked = g.sort_values("score", ascending=False)["option"].tolist()
        true_option = g.loc[g["label"] == 1, "option"].iloc[0]
        top3 = ranked[:3]
        scores.append(1.0 / (top3.index(true_option) + 1) if true_option in top3 else 0.0)
    return sum(scores) / len(scores)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))
eps = CONFIG["label_smoothing"]

def train_one_epoch():
    model.train()
    total_loss, n_batches = 0.0, 0
    for Xw, Xc, y in train_loader:
        Xw, Xc, y = Xw.to(device), Xc.to(device), y.to(device)
        y_smooth = y * (1 - eps) + 0.5 * eps   # 1 -> 1-eps/2, 0 -> eps/2

        with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
            logits = model(Xw, Xc)
            loss = criterion(logits, y_smooth)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches

@torch.no_grad()
def evaluate(loader, meta_df):
    model.eval()
    all_scores, total_loss, n_batches = [], 0.0, 0
    for Xw, Xc, y in loader:
        Xw, Xc, y = Xw.to(device), Xc.to(device), y.to(device)
        with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
            logits = model(Xw, Xc)
            loss = criterion(logits, y)
        total_loss += loss.item(); n_batches += 1
        all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    scored = meta_df.copy()
    scored["score"] = all_scores
    return total_loss / n_batches, compute_map3(scored)


In [9]:
best_val_map3 = 0.0
patience_left = CONFIG["early_stop_patience"]

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch()
    val_loss, val_map3 = evaluate(val_loader, meta_val)
    scheduler.step(val_map3)

    wandb.log({
        "epoch": epoch, "train/loss": train_loss,
        "val/loss": val_loss, "val/map3": val_map3,
        "lr": optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_val_map3 + CONFIG["early_stop_min_delta"]:
        best_val_map3 = val_map3
        patience_left = CONFIG["early_stop_patience"]
        torch.save(model.state_dict(), "best_pairwise_model.pt")
    elif epoch >= CONFIG["early_stop_min_epochs"]:
        patience_left -= 1

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} val_loss {val_loss:.4f} | val_map3 {val_map3:.4f} | patience {patience_left}")

    if epoch >= CONFIG["early_stop_min_epochs"] and patience_left <= 0:
        print(f"Early stopping: no val_map3 improvement for {CONFIG['early_stop_patience']} epochs (after warmup).")
        break

print("Best val MAP@3:", best_val_map3)


Epoch 01 | train_loss 0.4573 val_loss 0.4198 | val_map3 0.9744 | patience 8
Epoch 02 | train_loss 0.2174 val_loss 0.1160 | val_map3 0.9894 | patience 8
Epoch 03 | train_loss 0.1730 val_loss 0.0744 | val_map3 0.9978 | patience 8
Epoch 04 | train_loss 0.1553 val_loss 0.0659 | val_map3 0.9978 | patience 8
Epoch 05 | train_loss 0.1491 val_loss 0.0613 | val_map3 0.9978 | patience 8
Epoch 06 | train_loss 0.1464 val_loss 0.0567 | val_map3 0.9978 | patience 8
Epoch 07 | train_loss 0.1409 val_loss 0.0554 | val_map3 0.9978 | patience 8
Epoch 08 | train_loss 0.1369 val_loss 0.0495 | val_map3 0.9978 | patience 8
Epoch 09 | train_loss 0.1355 val_loss 0.0506 | val_map3 0.9978 | patience 8
Epoch 10 | train_loss 0.1347 val_loss 0.0472 | val_map3 0.9978 | patience 8
Epoch 11 | train_loss 0.1340 val_loss 0.0469 | val_map3 0.9978 | patience 8
Epoch 12 | train_loss 0.1332 val_loss 0.0460 | val_map3 0.9978 | patience 7
Epoch 13 | train_loss 0.1331 val_loss 0.0456 | val_map3 0.9978 | patience 6
Epoch 14 | t

In [10]:
model.load_state_dict(torch.load("best_pairwise_model.pt"))
model.eval()

all_scores = []
with torch.no_grad():
    for Xw, Xc in test_loader:
        Xw, Xc = Xw.to(device), Xc.to(device)
        logits = model(Xw, Xc)
        all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())

scored_test = test_long[["qid", "option"]].copy()
scored_test["score"] = all_scores

pred_rows = []
for qid, g in scored_test.groupby("qid", sort=False):
    top3 = g.sort_values("score", ascending=False)["option"].tolist()[:3]
    pred_rows.append({"ID": qid, "Prediction": " ".join(top3)})

submission = pd.DataFrame(pred_rows)
submission.to_csv("submission_pairwise.csv", index=False)
submission.head()


,ID,Prediction
0,1,A E B
1,2,B D E
2,3,B D A
3,4,E C D
4,5,C A D


In [11]:
wandb.finish()


epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
lr,██████▄▄▄▄▂▂▂▂▁▁▁▁▁
train/loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/map3,▁▅█████████████████
epoch,19
lr,6e-05
train/loss,0.12949
val/loss,0.04209
val/map3,0.99778
